[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/labs/lab-10-attention-seq2seq.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Lab 10 — Attention, and a translation system

**MSIS · AME 5053 · Week 10 · 3 hours**

Session 27 built an encoder-decoder and then showed where it breaks: the encoder squeezes a whole
sentence into one fixed vector, and the decoder sees nothing else. Session 28's answer was to stop
squeezing. Instead of one context vector <code>c</code> for the whole translation, the decoder
computes a fresh <code>c_i</code> at every step, as a weighted average over all the encoder's
states, with the weights decided by what the decoder is looking for right now.

This lab builds that. By the end you will have a working English-to-French translation system —
a small and imperfect one — and, more to the point, you will have the picture of what it attends
to while translating. Bahdanau's Figure 3, which session 28 asked you to look at properly, is a
plot of exactly the numbers this notebook computes.

**By the end of this lab you will be able to:**

1. Prepare a parallel corpus: two vocabularies, two padded tensors, and the `<sos>`/`<eos>` markers
2. Build a bidirectional GRU encoder and say why the annotations are twice as wide as the hidden state
3. Implement attention from SLP3's equations (13.35)–(13.37) as a function of three lines
4. Train a decoder with teacher forcing, and control it with `sample_probability`
5. Plot an alignment heat map and read what the model learned about word order

---

## Part 0 — A parallel corpus

Everything so far in this course has been one sentence at a time. Translation needs **pairs**: a
sentence and its translation, aligned. NLTK ships one — `comtrans`, drawn from the European
Parliament's proceedings, where every debate is published in each official language.

In [ ]:
%pip install -q nltk matplotlib

import nltk
ok = nltk.download("comtrans")
print("comtrans downloaded:", ok)

import random, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
print("torch", torch.__version__)
print("Done.")

### Where this will run

The lab works with or without a GPU. A GPU helps this one least of the three in Unit III — the
decoder loops one word at a time in Python, waiting on itself, which is a shape of work GPUs do not
accelerate much. Ask for one if it is offered, but the CPU path is the one these numbers came from.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("training on:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

> **Save your own copy now:** File → Save a copy in Drive.

Each aligned sentence has `.words` (English) and `.mots` (French — *mots* is simply French for
"words"). Look at one before going further.

In [ ]:
from nltk.corpus import comtrans

aligned = comtrans.aligned_sents("alignment-en-fr.txt")
print("aligned pairs:", len(aligned))
print("EN:", " ".join(aligned[1].words))
print("FR:", " ".join(aligned[1].mots))

# Verified output:
#   aligned pairs: 33334

### The length cap

Europarl sentences are long — the average pair is 20 English tokens and 22 French. Long sentences
are slow to train on and hard to learn from, so we keep only pairs where **both** sides are 15
tokens or shorter.

This is a real cost, and worth being clear about: it throws away three quarters of the corpus and
leaves the shortest, simplest sentences. Everything this lab concludes is about those.

In [ ]:
CAP = 15

pairs = [([w.lower() for w in s.words], [m.lower() for m in s.mots])
         for s in aligned
         if 1 <= len(s.words) <= CAP and 1 <= len(s.mots) <= CAP]

print(f"pairs kept: {len(pairs)} of {len(aligned)} ({len(pairs)/len(aligned)*100:.0f}%)")
print("EN:", " ".join(pairs[0][0]))
print("FR:", " ".join(pairs[0][1]))

# Verified output:
#   pairs kept: 8525 of 33334 (26%)

---

## Part 1 — Two vocabularies, and two special tokens

Lab 9 built one vocabulary. Translation needs two, because the two languages share almost no
words, and the decoder's output layer has to have one slot per French word.

Two new tokens appear here that lab 9 did not need. The decoder has to be told when to start —
that is `<sos>` — and it has to be able to say when it has finished, which is `<eos>`. Without
`<eos>` a decoder simply runs until it hits the length limit.

In [ ]:
from collections import Counter

PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ["<pad>", "<sos>", "<eos>", "<unk>"]

random.seed(42)
random.shuffle(pairs)
n_test = max(200, len(pairs) // 10)
train_pairs, test_pairs = pairs[:-n_test], pairs[-n_test:]

def build_vocab(sequences, min_count=2):
    """Word -> index, with the four special tokens first."""
    # YOUR CODE HERE
    # 1. Counter over every token of every sequence
    # 2. itos = SPECIALS + words appearing at least min_count times, most frequent first
    # 3. return {word: index}, itos
    pass

In [ ]:
en_stoi, en_itos = build_vocab([en for en, _ in train_pairs])
fr_stoi, fr_itos = build_vocab([fr for _, fr in train_pairs])

print(f"train {len(train_pairs)} · test {len(test_pairs)}")
print(f"English vocabulary {len(en_itos)} · French vocabulary {len(fr_itos)}")

# Verified output:
#   train 7673 · test 852
#   English vocabulary 3213 · French vocabulary 3678

The target sequence is wrapped in the two markers, so that a training example reads
`<sos> le débat est clos . <eos>`. The decoder is given the `<sos>` and asked for the next word,
given `<sos> le` and asked for the next, and so on — which is the same next-word objective
session 11 built n-grams around, now conditioned on an English sentence.

In [ ]:
def encode_pairs(pairs):
    src = [[en_stoi.get(t, UNK) for t in en] for en, _ in pairs]
    tgt = [[SOS] + [fr_stoi.get(t, UNK) for t in fr] + [EOS] for _, fr in pairs]
    return src, tgt

def batchify(src, tgt, idx):
    """Pad a batch of pairs. Returns source tensor, true source lengths, target tensor."""
    s = [src[i] for i in idx]
    t = [tgt[i] for i in idx]
    slen = torch.tensor([len(x) for x in s])
    S = torch.zeros(len(s), int(slen.max()), dtype=torch.long)
    T = torch.zeros(len(t), max(len(x) for x in t), dtype=torch.long)
    for r, (a, b) in enumerate(zip(s, t)):
        S[r, :len(a)] = torch.tensor(a)
        T[r, :len(b)] = torch.tensor(b)
    return S, slen, T

train_src, train_tgt = encode_pairs(train_pairs)
test_src, test_tgt = encode_pairs(test_pairs)

S, slen, T = batchify(train_src, train_tgt, [0, 1, 2])
print("source batch:", tuple(S.shape), "· target batch:", tuple(T.shape))
print("target row 0:", " ".join(fr_itos[i] for i in T[0].tolist()))

---

## Part 2 — The encoder, in both directions

Session 28 quoted Bahdanau's reason for reading the sentence in both directions: "we would like
the annotation of each word to summarize not only the preceding words, but also the following
words." Attention weights a *position*, so that position's vector had better summarise its whole
context, not just its left half.

In PyTorch this is one argument. `nn.GRU(..., bidirectional=True)` runs two GRUs, one over the
sentence forwards and one backwards, and concatenates them — so a hidden size of 256 produces
annotations of width 512.

That width matters in Part 3, so notice it now.

In [ ]:
HIDDEN, EMB = 256, 128

class Encoder(nn.Module):
    def __init__(self, vocab, emb=EMB, hidden=HIDDEN):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=PAD)
        self.rnn = nn.GRU(emb, hidden, batch_first=True, bidirectional=True)
        # The decoder starts from a summary of the whole sentence: the two final states.
        self.bridge = nn.Linear(hidden * 2, hidden)
        # SLP3's score is a dot product, which is only defined between vectors of equal
        # width. The annotations are 2H and the decoder's query is H, so project.
        self.project = nn.Linear(hidden * 2, hidden)

    def forward(self, x, lens):
        e = self.emb(x)
        packed = nn.utils.rnn.pack_padded_sequence(e, lens, batch_first=True, enforce_sorted=False)
        out, h = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        h0 = torch.tanh(self.bridge(torch.cat([h[0], h[1]], dim=1)))
        return self.project(out), h0

enc = Encoder(len(en_itos))
eo, h0 = enc(S, slen)
print("annotations:", tuple(eo.shape), "  (batch, source length, hidden)")
print("initial decoder state:", tuple(h0.shape))

### An aside worth having

That projection is not bookkeeping. Bahdanau's own score function is **additive** — it feeds the
annotation and the query through a small network, which works whatever their widths. SLP3's
(13.35) uses a **dot product**, which requires them to match. Session 28 named this disagreement
between the two sources; here is where it has a consequence in code, and projecting is the cheapest
way to keep SLP3's version.

---

## Part 3 — Attention

Three equations from session 28, and each is one line of PyTorch.

Score every encoder state against what the decoder is looking for, which is its current hidden
state <code>h_t</code>:

    scores = sum(annotations * query, dim=-1)

Turn the scores into weights that sum to one:

    alpha = softmax(scores)

Take the weighted average of the annotations:

    context = sum(annotations * alpha, dim=1)

That is (13.35), (13.36) and (13.37). The name below is Rao & McMahan's, kept so their chapter 8
reads directly against this notebook.

In [ ]:
def verbose_attention(encoder_state_vectors, query_vector, mask):
    """SLP3 (13.35)-(13.37).

    encoder_state_vectors : [batch, source_len, hidden]  — one annotation per source word
    query_vector          : [batch, hidden]              — the decoder's current state
    mask                  : [batch, source_len]          — True where a real word sits

    Returns (context [batch, hidden], alpha [batch, source_len]).
    """
    # YOUR CODE HERE
    # 1. scores: multiply the annotations by the query and sum over the last dimension
    # 2. mask out the padding with float("-inf") BEFORE the softmax, or padding gets weight
    # 3. alpha: softmax the scores over the source positions
    # 4. context: multiply the annotations by alpha and sum over the source positions
    pass

In [ ]:
context, alpha = verbose_attention(eo, h0, (S != PAD))
print("context:", tuple(context.shape), "· alpha:", tuple(alpha.shape))
print("alpha row 0 sums to:", round(alpha[0].sum().item(), 6))
assert torch.allclose(alpha.sum(1), torch.ones(alpha.size(0)), atol=1e-5)
assert alpha[0][slen[0]:].sum().item() == 0.0, "padding must receive no attention"
print("Attention weights are a distribution, and the padding gets none of it.")

The masking step is the one to be careful about. Padding positions hold real numbers in the
annotation tensor — the encoder computed something for them — so without the mask they receive
attention weight, and the context vector is contaminated by positions that are not words. Lab 9
made the same point about the final hidden state. It is the same bug wearing a different coat.

---

## Part 4 — The decoder, one step at a time

The decoder cannot use `nn.GRU` the way the encoder does. `nn.GRU` consumes a whole sequence in
one call, and this decoder has to stop at every step to compute attention against the encoder's
states. So we use `nn.GRUCell`, which is one step of the same arithmetic, and write the loop.

At each step: compute the context from where we are, feed the previous word and the context into
the cell, and predict the next French word from the new state and the context together.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab, emb=EMB, hidden=HIDDEN):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=PAD)
        self.cell = nn.GRUCell(emb + hidden, hidden)
        self.out = nn.Linear(hidden * 2, vocab)

    def forward(self, enc_out, h, target, mask, sample_probability=0.0):
        B, T = target.shape
        logits, alphas = [], []
        y = target[:, 0]                                   # <sos> for every row
        for t in range(1, T):
            context, alpha = verbose_attention(enc_out, h, mask)
            h = self.cell(torch.cat([self.emb(y), context], dim=1), h)
            step = self.out(torch.cat([h, context], dim=1))
            logits.append(step)
            alphas.append(alpha)

            # Teacher forcing, with a dial. sample_probability = 0 always feeds the gold
            # word; 1 always feeds the model's own prediction; in between is scheduled
            # sampling, which session 27 introduced by name.
            use_own = torch.rand(B, device=target.device) < sample_probability
            y = torch.where(use_own, step.argmax(1), target[:, t])

        return torch.stack(logits, 1), torch.stack(alphas, 1)

    @torch.no_grad()
    def greedy(self, enc_out, h, mask, max_len=16):
        """Translate with no gold answer available — what inference actually looks like."""
        B = enc_out.size(0)
        y = torch.full((B,), SOS, dtype=torch.long, device=enc_out.device)
        outs, alphas = [], []
        for _ in range(max_len):
            context, alpha = verbose_attention(enc_out, h, mask)
            h = self.cell(torch.cat([self.emb(y), context], dim=1), h)
            y = self.out(torch.cat([h, context], dim=1)).argmax(1)
            outs.append(y)
            alphas.append(alpha)
        return torch.stack(outs, 1), torch.stack(alphas, 1)

The two methods differ in one place only: `forward` may be handed the gold word, and `greedy`
never is. That is the gap session 27 described — the model is trained in a condition it never
meets at inference, standing on correct prefixes it will not have — and `sample_probability` is
the dial that narrows it.

---

## Part 5 — Training

Nothing in this loop is new. The one detail worth naming is `ignore_index=PAD`, which keeps the
loss from rewarding the model for predicting padding, of which there is a great deal.

In [ ]:
def token_accuracy(logits, target):
    """Fraction of non-padding target words predicted correctly, given the gold prefix."""
    pred = logits.argmax(2)
    gold = target[:, 1:]
    keep = gold != PAD
    return ((pred == gold) & keep).sum().item() / keep.sum().item()   # .item() syncs from any device


def train(epochs=6, bs=32, lr=1e-3, sample_probability=0.0, seed=42):
    torch.manual_seed(seed)
    encoder = Encoder(len(en_itos)).to(DEVICE)
    decoder = Decoder(len(fr_itos)).to(DEVICE)
    opt = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=lr)
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)
    t0 = time.time()

    for ep in range(epochs):
        encoder.train(); decoder.train()
        order = torch.randperm(len(train_src))
        for i in range(0, len(train_src), bs):
            idx = order[i:i + bs].tolist()
            Sb, sl, Tb = batchify(train_src, train_tgt, idx)
            Sb, Tb = Sb.to(DEVICE), Tb.to(DEVICE)      # sl stays on the CPU: pack needs it there
            opt.zero_grad()
            eo, h = encoder(Sb, sl)
            logits, _ = decoder(eo, h, Tb, (Sb != PAD), sample_probability)
            loss = loss_fn(logits.reshape(-1, logits.size(-1)), Tb[:, 1:].reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(decoder.parameters()), 5.0)
            opt.step()

        encoder.eval(); decoder.eval()
        with torch.no_grad():
            Sb, sl, Tb = batchify(test_src, test_tgt, list(range(len(test_src))))
            Sb, Tb = Sb.to(DEVICE), Tb.to(DEVICE)
            eo, h = encoder(Sb, sl)
            logits, _ = decoder(eo, h, Tb, (Sb != PAD))
            acc = token_accuracy(logits, Tb)
        print(f"  epoch {ep + 1}: test token accuracy {acc:.4f}  ({time.time() - t0:.0f}s)")

    return encoder, decoder

In [ ]:
encoder, decoder = train(epochs=6)

### What the number means, and what it does not

Token accuracy here is measured **with the gold prefix supplied** at every step, which is a much
easier question than translating from scratch. It is a training diagnostic, not a translation
score. The next cell is the honest test.

In [ ]:
Sb, sl, Tb = batchify(test_src, test_tgt, list(range(12)))
Sb, Tb = Sb.to(DEVICE), Tb.to(DEVICE)
eo, h = encoder(Sb, sl)
outs, alphas = decoder.greedy(eo, h, (Sb != PAD))
outs, alphas, Sb, Tb = outs.cpu(), alphas.cpu(), Sb.cpu(), Tb.cpu()   # back for printing/plotting

for r in range(6):
    n = int(sl[r])
    hyp = []
    for i in outs[r].tolist():
        if i == EOS:
            break
        hyp.append(fr_itos[i])
    print("EN  ", " ".join(en_itos[i] for i in Sb[r][:n].tolist()))
    print("gold", " ".join(fr_itos[i] for i in Tb[r].tolist() if i not in (PAD, SOS, EOS)))
    print("out ", " ".join(hyp), "\n")

Short sentences come out well; longer ones drift into plausible parliamentary noise. 8525
pairs is a very small corpus for translation, and this is what that looks like from the inside.
Lab 9 reached the same wall from the other side — 1,500 documents was not enough to train a
sentiment model — and session 32 is where the course answers it properly.

The alignment, though, does not need a large corpus to be worth looking at.

---

## Part 6 — The alignment map

This is Bahdanau's Figure 3, computed from your own model. Each row is one French word the
decoder produced; each column is an English word it was reading. A bright cell says that when
the decoder wrote this French word, it was looking at that English one.

In [ ]:
def plot_alignment(row=0):
    n = int(sl[row])
    hyp = []
    for i in outs[row].tolist():
        if i == EOS:
            break
        hyp.append(fr_itos[i])
    A = alphas[row][:len(hyp), :n].detach().numpy()

    fig, ax = plt.subplots(figsize=(1 + 0.55 * n, 1 + 0.45 * len(hyp)))
    ax.imshow(A, aspect="auto", cmap="Greys", vmin=0, vmax=1)
    ax.set_xticks(range(n), [en_itos[i] for i in Sb[row][:n].tolist()], rotation=60, ha="right")
    ax.set_yticks(range(len(hyp)), hyp)
    ax.set_xlabel("English (read by the encoder)")
    ax.set_ylabel("French (written by the decoder)")
    for edge in ("top", "right"):
        ax.spines[edge].set_visible(False)
    fig.tight_layout()
    plt.show()

plot_alignment(0)

### Reading it

Look for three things.

**A diagonal.** English and French largely share word order, so most of the mass should sit near
the diagonal. That is the model discovering alignment without ever being told what a word is —
nothing in the training data says which English word corresponds to which French one.

**Where the diagonal breaks.** It should break exactly where the two languages disagree about
order, which is what session 28 asked you to look for in Bahdanau's figure. French puts adjectives
after nouns; English puts them before.

**Whether the rows are sharp or soft.** Each row is a probability distribution over the English
words. A row with all its mass on one cell is a model that has committed; a spread-out row is a
model hedging. Part 7 is about that.

In [ ]:
# Try a few. Rows differ a lot — some are clean, some are a mess.
for r in (1, 2, 3):
    plot_alignment(r)

---

## Part 7 — The attention is sharper than it should be

Measure how concentrated the attention is. **Entropy** is the standard measure: it is 0 when all
the weight sits on one position, and <code>log n</code> when it is spread evenly over
<code>n</code> positions.

In [ ]:
def mean_attention_entropy(alphas, outs, lens, n_sentences=24):
    """Average entropy of the attention rows, over several decoded sentences.

    Low entropy means each output word looked at essentially one source word;
    high entropy means the weight was spread out. Uniform over n positions is log(n).
    """
    values = []
    for r in range(min(n_sentences, len(lens))):
        n = int(lens[r])
        length = 0
        for i in outs[r].tolist():
            if i == EOS:
                break
            length += 1
        if length < 3 or n < 4:
            continue
        a = alphas[r][:length, :n].detach().numpy()
        values.append(float(-(a * np.log(a + 1e-9)).sum(1).mean()))
    return float(np.mean(values))

Sb, sl, Tb = batchify(test_src, test_tgt, list(range(24)))
Sb, Tb = Sb.to(DEVICE), Tb.to(DEVICE)
eo, h = encoder(Sb, sl)
outs, alphas = decoder.greedy(eo, h, (Sb != PAD))
outs, alphas, Sb, Tb = outs.cpu(), alphas.cpu(), Sb.cpu(), Tb.cpu()

ent_unscaled = mean_attention_entropy(alphas, outs, sl)
print(f"mean entropy, unscaled scores : {ent_unscaled:.3f}")
print(f"uniform over 8 positions would be : {math.log(8):.3f}")

The measured entropy is far below uniform — the attention is close to one-hot, putting nearly all
of each row's weight on a single source word. That is not a property of attention; it is a property
of *this* score function, and it has a cause.

The score is a dot product of two 256-dimensional vectors. The more dimensions, the larger the
dot products get, and softmax of large numbers is nearly one-hot: one position takes essentially
all the weight and the others take almost none. The gradient through a saturated softmax is tiny,
so the model has trouble changing its mind about where to look.

The fix is to divide the scores by <code>sqrt(d)</code>, where <code>d</code> is the width of the
vectors being dotted. Session 29 introduces exactly this factor, for exactly this reason. Here it
is, one session early, as something you can measure.

In [ ]:
def scaled_attention(encoder_state_vectors, query_vector, mask):
    """(13.35) with the scores divided by sqrt(d) — session 29's factor."""
    d = encoder_state_vectors.size(-1)
    scores = torch.sum(encoder_state_vectors * query_vector.unsqueeze(1), dim=2) / math.sqrt(d)
    scores = scores.masked_fill(~mask, float("-inf"))
    alpha = F.softmax(scores, dim=1)
    context = torch.sum(encoder_state_vectors * alpha.unsqueeze(2), dim=1)
    return context, alpha

# Retrain with the scaled score. Same everything else.
verbose_attention = scaled_attention
encoder_s, decoder_s = train(epochs=6)

In [ ]:
eo, h = encoder_s(Sb.to(DEVICE), sl)
outs_s, alphas_s = decoder_s.greedy(eo, h, (Sb.to(DEVICE) != PAD))
outs_s, alphas_s = outs_s.cpu(), alphas_s.cpu()
ent_scaled = mean_attention_entropy(alphas_s, outs_s, sl)

print(f"unscaled entropy : {ent_unscaled:.3f}")
print(f"scaled entropy   : {ent_scaled:.3f}")
print(f"uniform would be : {math.log(8):.3f}")

# Verified output:
#   unscaled entropy : 0.153
#   scaled entropy   : 1.450

Scaling changes the *shape* of the attention substantially and the accuracy hardly at all:
entropy moves from 0.153 to 1.450, while token accuracy goes from
0.4581 to 0.4770.

Both halves of that are worth keeping. The accuracy says the saturation was not what was limiting
this model — the corpus was. The entropy says the mechanism was nonetheless working in a
degenerate regime, one where the softmax has become almost an argmax and the gradient through it
is nearly flat. At 256 dimensions it is survivable. Session 31's transformer runs attention at
larger widths and many heads at once, where it is not, which is why the scaling factor is written
into the definition there rather than offered as an improvement.

---

## Extension — the teacher-forcing dial

Session 27 introduced scheduled sampling: feed the model its own output some of the time during
training, so it gets practice standing on its own mistakes. The decoder already takes the
argument, so this costs one training run.

Measure it before assuming it helps. The idea is sound and well known, which is not the same as
it being an improvement on every corpus.

In [ ]:
# YOUR CODE HERE
# Train with sample_probability = 0.25 and compare against the 0.0 run above.
# It costs one training run, about a minute.

---

## What this lab established

1. A parallel corpus needs two vocabularies, and a decoder needs `<sos>` and `<eos>` to know where
   a sentence starts and stops.
2. Attention is three lines: score, softmax, weighted sum — and the masking step that stops the
   padding from taking weight.
3. The decoder loop runs one step at a time because it has to consult the encoder between steps.
4. The alignment map shows a model discovering which French word goes with which English one,
   without ever being told that words correspond.
5. Unscaled dot-product attention saturates, and <code>sqrt(d)</code> is the fix — which is where
   session 29 begins.

Lab 11 drops the recurrence entirely. Everything that made this decoder slow — one step at a time,
each waiting on the last — is what the transformer removes, keeping only the attention.